# Machine Learning: gec çatdırılmanı proqnozlaşdırmaq
**Məqsəd:** sifariş verilən anda bilinən məlumata görə sifarişin gec çatıb-çatmayacağını proqnozlaşdırmaq.

**Model:** Random Forest.

Statistika bölməsində gördük ki, gec çatdırılan sifarişlərin rəy balı xeyli aşağıdır (2.27, vaxtında çatdırılanlarda 4.29). Gec çatacaq sifarişi əvvəlcədən bilmək biznesə müdaxilə etməyə imkan verir.


Model yalnız sifariş verilən anda bilinən məlumatı görməlidir. Real çatdırılma tarixi və rəy balı sonradan məlum olur, ona görə əlamət ola bilməz. Sifarişlərin 93.3%-i vaxtında çatıb. Ona görə modeli yalnız dəqiqliklə yox, gec sifarişləri nə qədər tapdığı ilə də qiymətləndirəcəyik.


In [1]:
import sys
from pathlib import Path
import pandas as pd

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yml").exists())
sys.path.append(str(_root / "notebooks"))

from db import get_engine
engine = get_engine()

In [2]:
import sklearn
sklearn.__version__

'1.9.1'

### Step1

Silver qatındakı cədvəllərin sütunlarına baxırıq.Sifariş verilən anda məlum olan məlumatları seçmək. Çatdırılma baş verəndən sonra məlum olan şeyləri modelə verməyəcəyik, çünki bu "data leakage" yaradar.

In [3]:
pd.read_sql("SELECT TOP 1 * FROM silver.order_items", engine).columns.tolist()

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value']

In [4]:
pd.read_sql("SELECT TOP 1 * FROM silver.products", engine).columns.tolist()

['product_id',
 'category_name',
 'name_length',
 'description_length',
 'photos_qty',
 'weight_g',
 'length_cm',
 'height_cm',
 'width_cm']

 order_items cədvəlində qiymət, daşınma haqqı, satıcı və məhsul məlumatı var. products cədvəlində kateqoriya, çəki və ölçülər var. Bunların hamısı sifariş verilən anda məlumdur, ona görə modelə əlamət kimi verə bilərik.

In [5]:
pd.read_sql("SELECT TOP 1 * FROM silver.sellers", engine).columns.tolist()

['seller_id', 'zip_code_prefix', 'city', 'state']

In [6]:
pd.read_sql("SELECT TOP 1 * FROM silver.customers", engine).columns.tolist()

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

sellers cədvəlində satıcının indeksi, şəhəri və ştatı var. customers cədvəlində müştərinin indeksi, şəhəri və ştatı var. Satıcı və müştəri ştatları fərqlidirsə, məsafə böyükdür və gecikmə ehtimalı artır.

## Step2

Hər sifariş üçün is_late hədəfini düzəldirik. Real çatdırılma tarixi gözlənilən tarixdən gec olubsa 1, vaxtında olubsa 0.
Modelə nəyi proqnozlaşdıracağını göstəririk

In [8]:
pd.read_sql("SELECT TOP 1 * FROM silver.orders", engine).columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

silver.orders cədvəlindən statusu delivered olan və real çatdırılma tarixi boş olmayan sifarişlərə baxırıq.Sifarişin gec olub-olmadığını yalnız çatdırılıb bitmiş sifarişlərdə bilirik. Ləğv olunmuş və ya yolda olan sifarişin real tarixi yoxdur ona görə onlara gec və yaxudda vaxtında deyə bilmərik. 

In [9]:
orders = pd.read_sql("""
    SELECT order_id, customer_id, order_purchase_timestamp,
           order_delivered_customer_date, order_estimated_delivery_date
    FROM silver.orders
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
""", engine)
len(orders)

96470

Statusu delivered olan və real çatdırılma tarixi dolu olan 96 470 sifariş var. Modeli bu sifarişlər üzərində öyrədəcəyik.

 Real çatdırılma tarixini gözlənilən tarixlə müqayisə edirik. Real tarix gözləniləndən sonradırsa is_late = 1 gec, deyilsə is_late = 0 vaxtında. Sonra value_counts() ilə hər qrupun sayına baxırıq.
Tarixləri müqayisə etməzdən əvvəl tarix tipinə çeviririk, çünki mətn kimi müqayisə yanlış nəticə verə bilər.Real tarixdən saatı silirik normalize, çünki gözlənilən tarixdə saat yoxdur və günləri günlərlə müqayisə etmək düzgündür.Qrupların sayına baxırıq ki sinif balansını görək.

In [10]:
for c in ["order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c])

orders["is_late"] = (orders["order_delivered_customer_date"].dt.normalize()
                     > orders["order_estimated_delivery_date"]).astype(int)

orders["is_late"].value_counts()

is_late
0    89936
1     6534
Name: count, dtype: int64

 Vaxtında çatdırılan 0 olsn 89936 sifariş ,gec çatdırılan 1 olan 6534 sifariş.

## Step3

Modelə veriləcək əlamətləri hazırlayırıq.sifariş verilən anda məlum olan məlumatı istifadə edirik. Real çatdırılma tarixini və rəy balını əlamət kimi vermirik, çünki onlar çatdırılmadan sonra məlum olur və modelə cavabı əvvəlcədən deyər.

In [11]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])

orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month
orders["purchase_weekday"] = orders["order_purchase_timestamp"].dt.weekday
orders["estimated_days"] = (orders["order_estimated_delivery_date"]
                            - orders["order_purchase_timestamp"]).dt.days

orders[["purchase_month", "purchase_weekday", "estimated_days"]].describe()

,purchase_month,purchase_weekday,estimated_days
count,96470.000000,96470.000000,96470.000000
mean,6.031046,2.756494,23.372748
std,3.228479,1.967041,8.758421
min,1.000000,0.000000,2.000000
25%,3.000000,1.000000,18.000000
50%,6.000000,3.000000,23.000000
75%,8.000000,4.000000,28.000000
max,12.000000,6.000000,155.000000


Üç yeni əlamət düzəldildi və dəyərləri məntiqlidir.
purchase_month: 1-dən 12-yə qədər
purchase_weekday: 0-dan 6-ya qədər
estimated_days: gözlənilən çatdırılma müddəti, orta hesabla təxminən 23 gün 

Hər sifariş üçün məhsul sayı, ümumi qiymət, ümumi daşınma haqqı, ümumi çəki və ümumi həcmə baxıırq.Ağır, böyük və çox məhsullu sifarişlərin çatdırılması daha çətindir və daha tez gecikə bilər. Bu məlumatların hamısı sifariş anında məlumdur.

In [12]:
items = pd.read_sql("""
    SELECT oi.order_id,
           COUNT(*) AS item_count,
           SUM(oi.price) AS total_price,
           SUM(oi.freight_value) AS total_freight,
           SUM(p.weight_g) AS total_weight,
           SUM(p.length_cm * p.height_cm * p.width_cm) AS total_volume
    FROM silver.order_items oi
    JOIN silver.products p ON oi.product_id = p.product_id
    GROUP BY oi.order_id
""", engine)
items.describe()

,item_count,total_price,total_freight,total_weight,total_volume
count,98666.000000,98666.000000,98666.000000,98650.000000,9.865000e+04
mean,1.141731,137.754076,22.823562,2390.415307,1.740425e+04
std,0.538452,210.645145,21.650909,4773.529840,3.040383e+04
min,1.000000,0.850000,0.000000,0.000000,1.680000e+02
25%,1.000000,45.900000,13.850000,300.000000,2.964000e+03
50%,1.000000,86.900000,17.170000,750.000000,7.260000e+03
75%,1.000000,149.900000,24.040000,2068.750000,1.987200e+04
max,21.000000,13440.000000,1794.960000,184400.000000,1.476000e+06


98666 sifariş üçün məhsul əlamətləri hesablandı.Əksər sifarişdə 1 məhsul var maksimum 21.Orta qiymət təxminən 138, orta daşınma haqqı təxminən 23-dür.16 sifarişdə çəki və həcm məlumatı boşdur.Bəzi məhsulların çəkisi 0-dır. Say azdır ona görə dəyişməyə ehtiyac yoxdur.

Hər sifariş üçün müştərinin ştatı, satıcının ştatı və onların eyni olub-olmaması same_state: 1 = eyni, 0 = fərqli.
Məsafə çatdırılma müddətinə təsir edən əsas amillərdən biridir. Eyni ştatda çatdırılma qısa olur ştatlararası isə uzun. Bu məlumat da sifariş anında məlumdur.Bir sifarişdə bir neçə satıcı ola bilər. Sadəlik üçün sifariş başına bir satıcının ştatını götürürük.

In [13]:
geo = pd.read_sql("""
    SELECT o.order_id, c.customer_state, MIN(s.state) AS seller_state
    FROM silver.orders o
    JOIN silver.customers c ON o.customer_id = c.customer_id
    JOIN silver.order_items oi ON o.order_id = oi.order_id
    JOIN silver.sellers s ON oi.seller_id = s.seller_id
    GROUP BY o.order_id, c.customer_state
""", engine)

geo["same_state"] = (geo["customer_state"] == geo["seller_state"]).astype(int)
geo["same_state"].value_counts()

same_state
0    63256
1    35410
Name: count, dtype: int64

Müştəri və satıcı eyni ştatda 35410 sifariş 
Fərqli ştatda 63256 sifariş 

Sifarişlərin əksəriyyəti ştatlararası çatdırılır, yəni yol uzundur. same_state əlaməti modelə bu fərqi göstərəcək.

orders, items və geo  cədvəllərini birləşdiririk. Çəki və həcm boşluqlarını medianla doldururuq. Sonra əlamətləri X, hədəfi y kimi ayırırıq.Model bir cədvəldən öyrənir. Boş dəyərləri doldurmaq lazımdır. Median kənar dəyərlərdən təsirlənmir ona görə daha uyğundur.

In [14]:
data = orders.merge(items, on="order_id").merge(geo[["order_id", "same_state"]], on="order_id")

for c in ["total_weight", "total_volume"]:
    data[c] = data[c].fillna(data[c].median())

features = ["purchase_month", "purchase_weekday", "estimated_days",
            "item_count", "total_price", "total_freight",
            "total_weight", "total_volume", "same_state"]

X = data[features]
y = data["is_late"]

print(X.shape, X.isna().sum().sum())

(96470, 9) 0


Üç cədvəl birləşdirildi. Model üçün 96 470 sifariş və 9 əlamət hazırdır, boş dəyər yoxdur

## Step4

Məlumatı iki hissəyə bölürük: 80% öyrənmək üçün train, 20% yoxlamaq üçün test.

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(len(X_train), len(X_test))
print(y_train.mean().round(4), y_test.mean().round(4))

77176 19294
0.0677 0.0677


train hissəsi 77176 sifariş,
test hissəsi 19294 sifariş,
gec sifarişlərin payı hər iki hissədə 6,77%-dir.

## Step5

Random Forest modelini train hissəsində öyrədirik.Model çoxlu qərar ağacı qurur. Hər ağac əlamətlərə baxıb sifarişin gec olub-olmayacağını deyir.Gec sifarişlər azdır 6,8%, ona görə class_weight="balanced" yazırıq. Bu, modelə gec sifarişlərin səhvini daha ağır sayır və onları qaçırmamağı öyrədir.

In [16]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100, max_depth=10,
    class_weight="balanced", random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_feat

Random Forest modeli 77 176 sifarişdən ibarət train hissəsində öyrədildi.

## STep6

Modelin test hissəsində verdiyi proqnozları əsl cavablarla müqayisə edirik.Gec sifarişlər yalnız 6,8%-dir. Model hamısına vaxtında desə, 93% accuracy alar, amma bir gec sifarişi də tapmaz. Ona görə accuracy ilə yanaşı recall və precision-a da baxırıq.

Model test hissəsindəki hər sifariş üçün 0 vaxtında və ya 1 gec cavabı verir.
Bu cavabları sonrakı addımda əsl cavablarla müqayisə edəcəyik.

In [17]:
y_pred = model.predict(X_test)
pd.Series(y_pred).value_counts()

0    14440
1     4854
Name: count, dtype: int64

Model 19294 test sifarişindən 14440-na vaxtında (0), 4854-nə gec (1) deyib.

Əsl gec sifarişlərin sayı təxminən 1307-dir, model isə 4854 sifarişə gec deyib. Yəni model gec sifarişləri qaçırmamaq üçün şübhəli sifarişlərin çoxuna gec deyir. 

Modelin proqnozlarını əsl cavablarla müqayisə edib hər sinif üçün precision, recall və f1-score hesablayırıq.
Accuracy tək başına yanıldıcıdır, çünki sifarişlərin 93%-i vaxtında çatır.
Precision modelin "gec" dediyi sifarişlərin neçə faizi həqiqətən gec olub.
Recall əsl gec sifarişlərin neçə faizini model tapıb.
F1-score precision və recall-un ortası.

In [18]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=["vaxtında", "gec"]))

              precision    recall  f1-score   support

    vaxtında       0.96      0.77      0.86     17987
         gec       0.16      0.59      0.25      1307

    accuracy                           0.76     19294
   macro avg       0.56      0.68      0.55     19294
weighted avg       0.91      0.76      0.82     19294



gec sifarişlər üçün
Recall = 0,59 model əsl gec sifarişlərin təxminən 59%-ni tapır.
Precision = 0,16 model gec dediyi sifarişlərin yalnız 16%-də haqlıdır, qalanı yanlış həyəcan siqnalıdır.
Accuracy = 0,76 ümumi dəqiqlik. Hamısı vaxtında deyən model 93% accuracy alardı, amma heç bir gec sifarişi tapmazdı. Ona görə bu rəqəm tək başına əhəmiyyətli deyil.

 Model gec sifarişlərin yarıdan çoxunu aşkar edir, amma çox yanlış həyəcan verir. 9 sadə əlamətlə gecikmənin bütün səbəblərini hava, bayram, satıcının gecikməsi bilmək mümkün deyil. Model bunu tam həll etmir, amma riskli sifarişləri əvvəlcədən seçmək üçün faydalıdır.

Confusion matrix. Sətirlər əsl cavabdır, sütunlar modelin cavabıdır.
Precision və recall-un arxasındakı sifariş sayını görmək üçün. Model neçə gec sifarişi tapıb, neçəsini qaçırıb, neçəsinə yanlış gec deyib.

In [19]:
from sklearn.metrics import confusion_matrix

pd.DataFrame(confusion_matrix(y_test, y_pred),
             index=["əsl: vaxtında", "əsl: gec"],
             columns=["model: vaxtında", "model: gec"])

,model: vaxtında,model: gec
əsl: vaxtında,13900,4087
əsl: gec,540,767


767 gec sifariş düzgün tapıldı.
540 gec sifariş qaçırıldı model vaxtında dedi.
4087 vaxtında sifarişə yanlış gec deyildi (yanlış həyəcan).
13 900 vaxtında sifariş düzgün tanındı.

Gec sifarişlərin 767 / 1307 = 59%-i tapılıb (recall). Modelin gec dediyi 4854 sifarişdən yalnız 767-si həqiqətən gecdir (precision 16%). Model gecikmə riski olan sifarişləri seçməyə kömək edir, amma hər həyəcan siqnalı gecikmə demək deyil.

## Step7

Modelin hansı əlamətə daha çox əsaslandığına baxırıq.
Bu, gecikməyə təsir edən amilləri başa düşməyə kömək edir və modelin nəticəsini izah etməyə imkan verir.

In [20]:
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
importance.round(3)

purchase_month      0.357
estimated_days      0.184
total_freight       0.126
total_price         0.083
total_volume        0.075
total_weight        0.069
same_state          0.063
purchase_weekday    0.028
item_count          0.015
dtype: float64

Modelin proqnozda ən çox əsaslandığı əlamətlər:
1. Sifariş ayı 0,357: gecikmələr müəyyən aylarda (məsələn, bayram və kampaniya dövrlərində) daha çox olur.
2. Gözlənilən müddət 0,184: vəd olunan müddət qısadırsa, gecikmə ehtimalı artır.
3. Daşınma haqqı 0,126: bahalı daşınma uzaq və ya çətin çatdırılmanı göstərir.

Məhsul sayı (0,015) və həftənin günü (0,028) demək olar ki, təsir etmir.

Sifariş ayının bu qədər güclü olması modelin məlumatdakı konkret dövrlərin gecikmələrinə  bağlı ola biləcəyini göstərir. Başqa illər üçün nəticə fərqli ola bilər.

Model `models/late_delivery_rf.pkl` faylına saxlandı . Streamlit bu faylı yükləyib yeni sifarişlər üçün proqnoz verəcək.

In [22]:
import joblib

models_dir = _root / "models"
models_dir.mkdir(exist_ok=True)

joblib.dump(model, models_dir / "late_delivery_rf.pkl")

['c:\\Users\\Dell\\OneDrive\\Desktop\\olist-ecommerce-analytics\\models\\late_delivery_rf.pkl']

In [23]:
result = pd.DataFrame({
    "order_id": data["order_id"],
    "late_probability": model.predict_proba(X)[:, 1].round(4),
    "predicted_late": model.predict(X),
    "actual_late": y,
    "is_test": X.index.isin(X_test.index).astype(int)
})
result.head()

,order_id,late_probability,predicted_late,actual_late,is_test
0,e481f51cbdc54678b7cc49136f2d6af7,0.3093,0,0,0
1,53cdb2fc8bc7dce0b6741e2150273451,0.4113,0,0,0
2,47770eb9100c2d0c44946d9cf07ec65d,0.3132,0,0,1
3,949d5b44dbf5de918fe9c16f97b45f8a,0.6153,1,0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,0.1626,0,0,0


In [24]:
result.to_sql("ml_late_predictions", engine, schema="gold", if_exists="replace", index=False)

pd.read_sql("SELECT COUNT(*) AS n FROM gold.ml_late_predictions", engine)

,n
0,96470


### Yekun nəticə

96 470 çatdırılmış sifariş. Yalnız sifariş anında məlum olan 9 əlamət istifadə edildi . Real çatdırılma tarixi və rəy balı verilmədi data leakage yoxdur.
Random Forest  `class_weight="balanced"`. Gec sifarişlər cəmi 6,8% olduğu üçün train və test hissəsində eyni nisbət saxlanıldı (`stratify`).

- Recall 0,59: gec sifarişlərin təxminən 59%-i tapıldı.
- Precision 0,16: "gec" proqnozlarının yalnız 16%-i doğru çıxdı.
- Accuracy 0,76.

sifariş ayı 0,357, gözlənilən müddət 0,184, daşınma haqqı 0,126.

- Model yalnız 9 sadə əlamətə əsaslanır, hava, bayram və satıcının gecikməsi kimi amillər yoxdur.
- Çox yanlış həyəcan siqnalı verir (precision aşağıdır).
- Sifariş ayının güclü təsiri modelin məlumatdakı konkret dövrlərə (təxminən 2 il) bağlı ola biləcəyini göstərir.

**Çıxışlar:** `models/late_delivery_rf.pkl` (Streamlit üçün) və `gold.ml_late_predictions` (Power BI üçün).